# Module 9: RLAIF — PPO & GRPO
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jingyaogong/minimind-colab/blob/master/notebooks/09_rl_ppo_grpo.ipynb)

## Overview
This module covers **Reinforcement Learning from AI Feedback (RLAIF)** — training LLMs using RL without human labellers.

### RLHF vs RLAIF
| | RLHF | RLAIF |
|---|---|---|
| Reward signal | Human preferences | AI model scores |
| Cost | Expensive (labellers) | Cheap (automated) |
| Scale | Hard to scale | Scales easily |
| Quality | High (expert) | Depends on reward model |

### PPO (Proximal Policy Optimization)
- **Actor-critic** architecture: policy network + value network
- **GAE** (Generalized Advantage Estimation): `A_t = Σ (γλ)^k δ_{t+k}`
- **Clipped objective**: `L = E[min(r_t A_t, clip(r_t, 1-ε, 1+ε) A_t)]`
- Requires **two models in memory** (actor + critic) — memory-intensive

### GRPO (Group Relative Policy Optimization)
- **No critic network** — advantage computed from *group* of responses
- For a prompt x, generate G responses {y₁,...,y_G}, then:
  `A_i = (r_i - mean(r)) / std(r)`
- Much more **memory-efficient** than PPO — ideal for Colab

> ⚠️ **Note:** PPO needs 2 large models simultaneously and may OOM on Colab free tier.  GRPO is recommended for constrained environments.


In [ ]:
# ── Setup ──
import subprocess, sys

# Clone repo if not present
import os
if not os.path.exists('/content/minimind-colab'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/jingyaogong/minimind-colab.git',
                    '/content/minimind-colab'], check=True)

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                '/content/minimind-colab/requirements.txt'], check=True)

sys.path.insert(0, '/content/minimind-colab')
print("✅ Setup complete")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo "CPU only"


## 📊 Unified RL Framework

| Algorithm | # Models | Advantage Estimate | Notes |
|---|---|---|---|
| DPO | 1 (forward ×2) | implicit | Off-policy, no rollout needed |
| PPO | 2 | R - V(s) | Actor-critic, online, expensive |
| GRPO | 1 | (R - μ)/σ | Group-relative, memory efficient |

GRPO is the **default RL algorithm in MiniMind** because:
1. Only one model needed (vs two for PPO)
2. Advantage is naturally normalised within each group
3. No value-function bias or bootstrapping error


## 📥 Download RLAIF Dataset & Reward Model


In [ ]:
import os
os.makedirs('/content/minimind-colab/dataset', exist_ok=True)

# Download RLAIF dataset (~small JSONL file)
!modelscope download --dataset gongjy/minimind_dataset rlaif.jsonl \
    --local_dir /content/minimind-colab/dataset
print("Dataset downloaded!")

# Note: The reward model (internlm2-1_8b-reward) is large (~4 GB)
# For this course we use a lightweight proxy reward function instead.
print("Note: Using lightweight reward proxy for Colab compatibility")


## ⚙️ GRPO Configuration


In [ ]:
import argparse, torch

args = argparse.Namespace(
    save_dir='/content/minimind-colab/out',
    save_weight='grpo',
    from_weight='full_sft',
    data_path='/content/minimind-colab/dataset/rlaif.jsonl',
    hidden_size=768, num_hidden_layers=8, use_moe=0,
    epochs=1, batch_size=2, learning_rate=1e-6,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    dtype='bfloat16', num_workers=2,
    accumulation_steps=1, grad_clip=1.0,
    log_interval=20, save_interval=200,
    max_seq_len=256, max_gen_len=128,
    num_generations=4,   # G responses per prompt
    beta=0.04,           # KL penalty coefficient
    clip_eps=0.2,        # PPO clip epsilon
)
print(f"GRPO config loaded. Device: {args.device}")
print(f"  num_generations (G): {args.num_generations}")
print(f"  KL beta: {args.beta}  |  clip_eps: {args.clip_eps}")


## 🔄 GRPO Algorithm Explained

```
For each batch of prompts:
  1. Generate G=4 responses per prompt using policy π_θ
  2. Score each response with reward model r(x, y)
  3. Group advantage:  A_i = (r_i - mean(r)) / std(r)   ← No critic!
  4. Compute policy gradient with clipped ratio:
       L = E[ min(ratio * A, clip(ratio, 1-ε, 1+ε) * A) ]
  5. Add KL penalty:  KL(π_θ || π_ref)
  6. Update policy
```

**Key insight:** By comparing G responses *from the same prompt*, GRPO automatically  
normalises for prompt difficulty without a separate value network.


In [ ]:
import re

def simple_reward(response, prompt=""):
    """
    Lightweight reward function for Colab (no heavy reward model).
    In production use LMForRewardModel with internlm2-1_8b-reward.
    """
    reward = 0.0

    # Length reward: prefer responses 50-500 chars
    length = len(response.strip())
    if 50 <= length <= 500:
        reward += 0.5
    elif length < 20:
        reward -= 0.5

    # Repetition penalty
    words = response.split()
    if len(words) > 5:
        unique_ratio = len(set(words)) / len(words)
        reward += (unique_ratio - 0.5) * 0.5

    # Coherence: reward complete sentences
    if response.strip().endswith(('。', '！', '？', '.', '!', '?')):
        reward += 0.2

    return max(-1.0, min(1.0, reward))

# Test the reward function
test_responses = [
    "人工智能是一项改变世界的技术，它通过机器学习来模拟人类智能。",
    "好的好的好的好的好的好的好的",
    "是",
]
for r in test_responses:
    print(f"Response: {r[:60]!r}")
    print(f"Reward:   {simple_reward(r):.3f}\n")


In [ ]:
import copy, sys
sys.path.insert(0, '/content/minimind-colab')
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import RLAIFDataset
from trainer.trainer_utils import setup_seed, init_model, SkipBatchSampler
from torch import optim
from torch.utils.data import DataLoader
from contextlib import nullcontext

setup_seed(42)
lm_config = MiniMindConfig(hidden_size=args.hidden_size, num_hidden_layers=args.num_hidden_layers)
model, tokenizer = init_model(
    lm_config, args.from_weight,
    tokenizer_path='/content/minimind-colab/model',
    save_dir=args.save_dir, device=args.device
)

# Freeze reference model
ref_model = copy.deepcopy(model)
for p in ref_model.parameters():
    p.requires_grad = False
ref_model.eval()

train_ds = RLAIFDataset(args.data_path, tokenizer, max_length=args.max_seq_len)

optimizer = optim.AdamW(model.parameters(), lr=args.learning_rate)
scaler    = torch.cuda.amp.GradScaler(enabled=(args.dtype != 'float32' and 'cuda' in args.device))
dtype_t   = torch.bfloat16 if args.dtype == 'bfloat16' else torch.float16
autocast_ctx = (nullcontext() if 'cpu' in args.device
                else torch.cuda.amp.autocast(dtype=dtype_t))

print(f"Policy model params:    {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Reference model params: {sum(p.numel() for p in ref_model.parameters()) / 1e6:.1f}M  (frozen)")
print(f"Dataset: {len(train_ds)} prompts")


## 🏋️ GRPO Training Loop


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.notebook import tqdm
import torch.nn.functional as F

loss_history, reward_history, step_history = [], [], []


def grpo_train(epochs=1):
    model.train()
    global_step = 0

    for epoch in range(epochs):
        indices = list(range(len(train_ds)))
        batch_sampler = SkipBatchSampler(indices, args.batch_size, 0)
        loader = DataLoader(train_ds, batch_sampler=batch_sampler,
                            num_workers=0, pin_memory=False)
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for step, batch in enumerate(pbar, start=1):
            global_step += 1
            prompts = batch['prompt']   # list[str]

            prompt_inputs = tokenizer(
                prompts, return_tensors="pt", padding=True,
                padding_side="left", add_special_tokens=False
            ).to(args.device)

            # ── 1. Generate G responses per prompt ──────────────────────────
            all_output_ids, all_texts = [], []
            model.eval()
            with torch.no_grad():
                for _ in range(args.num_generations):
                    out = model.generate(
                        prompt_inputs["input_ids"],
                        attention_mask=prompt_inputs["attention_mask"],
                        max_new_tokens=args.max_gen_len, do_sample=True, temperature=0.8,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id
                    )
                    all_output_ids.append(out)
                    for j in range(out.shape[0]):
                        text = tokenizer.decode(
                            out[j][prompt_inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True
                        )
                        all_texts.append(text)
            model.train()

            # ── 2. Score responses ──────────────────────────────────────────
            # Production: use LMForRewardModel.get_score(prompt, response)
            B, G = len(prompts), args.num_generations
            rewards = torch.tensor([
                simple_reward(all_texts[i * G + j], prompts[i])
                for i in range(B) for j in range(G)
            ], device=args.device, dtype=torch.float32)

            # ── 3. Group advantages: A_i = (r_i - μ) / (σ + ε) ────────────
            rewards_grouped = rewards.view(B, G)
            means  = rewards_grouped.mean(1, keepdim=True)
            stds   = rewards_grouped.std(1, keepdim=True) + 1e-8
            advantages = ((rewards_grouped - means) / stds).view(-1)

            # ── 4. Policy update ────────────────────────────────────────────
            total_loss, count = torch.tensor(0.0, device=args.device), 0
            for i in range(B):
                for j in range(G):
                    idx        = i * G + j
                    output_ids = all_output_ids[j][i:i+1]
                    adv        = advantages[idx]

                    with autocast_ctx:
                        out_logits = model(output_ids).logits[:, :-1, :]
                        log_probs  = F.log_softmax(out_logits, dim=-1)
                        labels     = output_ids[:, 1:]
                        token_lp   = torch.gather(log_probs, 2,
                                                  labels.unsqueeze(-1)).squeeze(-1)

                        with torch.no_grad():
                            ref_logits  = ref_model(output_ids).logits[:, :-1, :]
                            ref_lp_all  = F.log_softmax(ref_logits, dim=-1)
                            ref_token_lp = torch.gather(ref_lp_all, 2,
                                                        labels.unsqueeze(-1)).squeeze(-1)

                        log_ratio      = token_lp - ref_token_lp
                        ratio          = torch.exp(log_ratio.clamp(-20, 20))   # clamp for stability
                        clipped_ratio  = torch.clamp(ratio, 1 - args.clip_eps, 1 + args.clip_eps)
                        pg_loss        = -torch.min(ratio * adv, clipped_ratio * adv).mean()
                        kl_loss        = log_ratio.mean()
                        loss           = pg_loss + args.beta * kl_loss

                    scaler.scale(loss).backward()
                    total_loss += loss.detach()
                    count      += 1

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            avg_loss   = total_loss.item() / max(count, 1)
            avg_reward = rewards.mean().item()
            loss_history.append(avg_loss)
            reward_history.append(avg_reward)
            step_history.append(global_step)
            pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'reward': f'{avg_reward:.3f}'})

            if global_step % args.log_interval == 0:
                clear_output(wait=True)
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
                ax1.plot(step_history, loss_history, 'b-', alpha=0.7)
                ax1.set_title('GRPO Loss'); ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
                ax2.plot(step_history, reward_history, 'g-', alpha=0.7)
                ax2.set_title('Average Reward'); ax2.set_xlabel('Step'); ax2.set_ylabel('Reward')
                plt.tight_layout(); plt.show()

        # Save checkpoint
        os.makedirs(args.save_dir, exist_ok=True)
        ckp = f'{args.save_dir}/{args.save_weight}_{lm_config.hidden_size}.pth'
        torch.save({k: v.half().cpu() for k, v in model.state_dict().items()}, ckp)
        print(f"\n✅ Checkpoint saved: {ckp}")


grpo_train(args.epochs)


In [ ]:
# Final reward/loss curves
if loss_history:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.plot(step_history, loss_history, 'b-', alpha=0.8)
    ax1.set_title('GRPO Training Loss'); ax1.set_xlabel('Step'); ax1.set_ylabel('Loss'); ax1.grid(True, alpha=0.3)
    ax2.plot(step_history, reward_history, 'g-', alpha=0.8)
    ax2.set_title('Average Group Reward'); ax2.set_xlabel('Step'); ax2.set_ylabel('Reward'); ax2.grid(True, alpha=0.3)
    plt.suptitle('GRPO Training Summary', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f"Final loss: {loss_history[-1]:.4f}  |  Final reward: {reward_history[-1]:.3f}")


## 📝 Advanced: PPO (Optional)

PPO requires a **critic (value) network** in addition to the policy network.  
For a 768-hidden MiniMind (~64M params), this means holding two 64M models in GPU memory simultaneously — likely causing OOM on a free Colab T4 (15 GB).

**Key differences from GRPO:**

```
PPO advantage:  A_t = r_t + γ V(s_{t+1}) - V(s_t)   (GAE, needs value net)
GRPO advantage: A_i = (r_i - mean(r)) / std(r)        (group stats, no value net)
```

To experiment with PPO locally (large GPU):
```python
# See trainer/trainer_ppo.py for the full implementation
# python trainer/trainer_ppo.py --hidden_size 512 --epochs 1
```

For Colab, **GRPO is the recommended approach** — same theoretical motivation, lower memory footprint.


In [ ]:
# Memory cleanup
import gc, torch
del ref_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory freed. Remaining: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")
else:
    print("CPU environment — no GPU memory to free")


## 📝 Student Exercise

1. **Reward shaping:** Modify `simple_reward` to add a factual-accuracy bonus  
   (e.g., check if numbers in the response are correct).
2. **Ablation:** Run GRPO with `num_generations=2` vs `num_generations=8`.  
   How does group size affect stability?
3. **KL coefficient:** Try `beta=0.0` (pure PG) and `beta=0.1`.  
   What happens to the reward trajectory?
4. **Clip epsilon:** GRPO with `clip_eps=0.1` vs `clip_eps=0.4` — plot the ratio distributions.

## 💡 Discussion Questions

- Why does GRPO *not* need a value network, while PPO does?  
- What are the failure modes of group-relative normalisation when G is very small?  
- How would you extend GRPO to handle multi-turn conversations?  
- What is the role of the KL penalty, and what happens if you remove it entirely?
